# ETL — Lending Club → Neon Postgres (Dimensional Model)

Loads resolved loans, builds a star schema (fact_loans + dimensions),
and writes to Neon. Reads credentials from ../.env (gitignored).


In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

DATA_PATH = "../data/raw/loan.csv"

cols_to_load = [
    "loan_amnt", "term", "int_rate", "installment", "grade", "sub_grade",
    "emp_length", "home_ownership", "annual_inc", "verification_status",
    "issue_d", "purpose", "addr_state",
    "dti", "delinq_2yrs", "earliest_cr_line",
    "inq_last_6mths", "open_acc", "pub_rec", "revol_bal", "revol_util", "total_acc",
    "mort_acc", "pub_rec_bankruptcies",
    "loan_status",
]

df = pd.read_csv(DATA_PATH, usecols=cols_to_load, low_memory=False)
df = df[df["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])].copy()
df["default"] = df["loan_status"].isin(["Charged Off", "Default"]).astype(int)
print("Working set:", df.shape)